In [4]:
!pip install kafka-python


  Obtaining dependency information for kafka-python from https://files.pythonhosted.org/packages/4a/db/694fd552295ed091e7418d02b6268ee36092d4c93211136c448fe061fe32/kafka_python-2.3.0-py2.py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 4.9 MB/s eta 0:00:00a 0:00:01


In [3]:
from kafka import KafkaProducer
producer = KafkaProducer(bootstrap_servers="kafka:9092")
print("✅ Connexion Kafka OK")


✅ Connexion Kafka OK


In [ ]:
from kafka import KafkaProducer
import requests, json, time

API_URL = "https://api.open-meteo.com/v1/forecast"
LAT, LON = 52.52, 13.41
KAFKA_TOPIC = "weather_transformed"
KAFKA_BROKER = "kafka:9092"

def fetch_weather():
    params = {
        "latitude": LAT,
        "longitude": LON,
        "current_weather": "true"
    }
    r = requests.get(API_URL, params=params, timeout=10)
    r.raise_for_status()
    return r.json().get("current_weather", {})

def transform_weather(record):
    if "temperature" in record:
        record["temp_f"] = record["temperature"] * 9/5 + 32
    record["high_wind_alert"] = record.get("windspeed", 0) > 10
    return record

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

while True:
    weather = fetch_weather()
    if weather:
        producer.send(KAFKA_TOPIC, transform_weather(weather))
        producer.flush()
        print("Sent:", weather)
    time.sleep(30)


Sent: {'time': '2026-01-22T11:00', 'interval': 900, 'temperature': -2.9, 'windspeed': 8.0, 'winddirection': 72, 'is_day': 1, 'weathercode': 0, 'temp_f': 26.78, 'high_wind_alert': False}
Sent: {'time': '2026-01-22T11:00', 'interval': 900, 'temperature': -2.9, 'windspeed': 8.0, 'winddirection': 72, 'is_day': 1, 'weathercode': 0, 'temp_f': 26.78, 'high_wind_alert': False}
Sent: {'time': '2026-01-22T11:00', 'interval': 900, 'temperature': -2.9, 'windspeed': 8.0, 'winddirection': 72, 'is_day': 1, 'weathercode': 0, 'temp_f': 26.78, 'high_wind_alert': False}
Sent: {'time': '2026-01-22T11:00', 'interval': 900, 'temperature': -2.9, 'windspeed': 8.0, 'winddirection': 72, 'is_day': 1, 'weathercode': 0, 'temp_f': 26.78, 'high_wind_alert': False}
Sent: {'time': '2026-01-22T11:00', 'interval': 900, 'temperature': -2.9, 'windspeed': 8.0, 'winddirection': 72, 'is_day': 1, 'weathercode': 0, 'temp_f': 26.78, 'high_wind_alert': False}
Sent: {'time': '2026-01-22T11:00', 'interval': 900, 'temperature': -2.

In [1]:
import socket

host = "spark"  # nom du service Docker Compose
port = 7077

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(5)

try:
    s.connect((host, port))
    print(f"✅ Connexion réussie à {host}:{port}")
except Exception as e:
    print(f"❌ Impossible de joindre {host}:{port}\n", e)
finally:
    s.close()


✅ Connexion réussie à spark:7077


In [3]:
pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 1.4 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
  Obtaining dependency information for py4j<0.10.9.10,>=0.10.9.7 from https://files.pythonhosted.org/packages/bd/db/ea0203e495be491c85af87b66e37acfd3bf756fd985f87e46fc5e3bf022c/py4j-0.10.9.9-py2.py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 286.5 kB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008644 sha256=08a329b525683e0f5cfdf888991d863fc27ba20e6175d71a4aa9f9f8360ad172
  Stored in directory: /home/jovyan/.cache/pip/wheels/16/33/a9/f8bff354a182417214933df74dace2a34b02c3e5643e8fac74
Successfully built pyspark
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WeatherAggregation") \
    .master("spark://spark:7077") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .getOrCreate()

print(spark.version)


RuntimeError: Java gateway process exited before sending its port number

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window, avg, count
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("WeatherAggregation") \
    .master("spark://spark:7077") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1") \
    .getOrCreate()

schema = StructType([
    StructField("temperature", DoubleType()),
    StructField("windspeed", DoubleType()),
    StructField("temp_f", DoubleType()),
    StructField("high_wind_alert", BooleanType()),
    StructField("time", StringType())
])

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "weather_transformed") \
    .load()

parsed = df.select(from_json(col("value").cast("string"), schema).alias("data")) \
           .select("data.*") \
           .withColumn("event_time", col("time").cast("timestamp"))

agg = parsed.groupBy(
    window(col("event_time"), "1 minute")
).agg(
    avg("temperature").alias("avg_temp_c"),
    count(col("high_wind_alert")).alias("alert_count")
)

query = agg.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

# agg.writeStream \
#     .format("csv") \
#     .option("path", "hdfs://namenode:9000/user/jovyan/weather_agg") \
#     .option("checkpointLocation", "/tmp/weather_checkpoint") \
#     .outputMode("append") \
#     .start()


RuntimeError: Java gateway process exited before sending its port number